<a href="https://colab.research.google.com/github/catdlia/GCI-world/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Імпорти та Базове налаштування

In [37]:
!pip install catboost optuna xgboost scikit-learn

import os
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

# Моделі
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Валідація
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# Шляхи
DATA_DIR = Path('/content/drive/MyDrive/GCI/competition')
train = pd.read_csv(DATA_DIR / 'input/train.csv')
test = pd.read_csv(DATA_DIR / 'input/test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'input/sample_submission.csv')

y = train['Drafted']
X_train_raw = train.drop(['Drafted'], axis=1)

Глобальний Препроцесинг та Доменні фічі (Domain Knowledge)

In [38]:
# Об'єднуємо для обробки
X_train_raw['is_train'] = 1
test['is_train'] = 0
df = pd.concat([X_train_raw, test], ignore_index=True)

# 1. Заповнення категорій та угруповання рідкісних шкіл
cat_features = ['School', 'Position_Type', 'Position', 'Player_Type']
for col in cat_features:
    df[col] = df[col].fillna('Unknown').astype(str)

school_counts = df['School'].value_counts()
rare_schools = school_counts[school_counts < 5].index
df.loc[df['School'].isin(rare_schools), 'School'] = 'Rare_School'
df['School_Raw'] = df['School'] # Зберігаємо для CatBoost

# 2. Магія пропусків
num_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
df['Total_NaNs'] = df[num_cols].isnull().sum(axis=1)
for col in num_cols:
    df[f'is_missing_{col}'] = df[col].isnull().astype(int)

# 3. Ваші "Золоті" доменні фічі
df['BMI'] = df['Weight'] / ((df['Height'] + 1e-6) ** 2)
df['Speed_Score'] = (df['Weight'] * 200) / ((df['Sprint_40yd'] + 1e-6) ** 4)
df['Explosion'] = df['Vertical_Jump'] + df['Broad_Jump']
df['Jump_Power'] = df['Vertical_Jump'] * df['Weight']

# 4. Pos_Diff (Найкраще для всіх дерев)
for col in num_cols + ['Weight', 'Height', 'BMI']:
    df[f'{col}_Pos_Diff'] = df.groupby('Position_Type')[col].transform(lambda x: x - x.mean()).fillna(0)

# Чистка перед навчанням
cols_to_drop = ['is_train', 'Id', 'Player_Type']
X_train_clean = df[df['is_train'] == 1].drop(cols_to_drop, axis=1)
X_test_clean = df[df['is_train'] == 0].drop(cols_to_drop, axis=1)

# Типизація
for col in ['Position_Type', 'Position', 'School_Raw']:
    X_train_clean[col] = X_train_clean[col].astype('category')
    X_test_clean[col] = X_test_clean[col].astype('category')

print(f"Готово! Розмірність трейну: {X_train_clean.shape}")

Готово! Розмірність трейну: (2781, 34)


Налаштування моделей та Monotonic Constraints

In [39]:
mono_constraints = {
    'Sprint_40yd': -1, 'Vertical_Jump': 1, 'Broad_Jump': 1,
    'Bench_Press_Reps': 1, 'Explosion': 1, 'Jump_Power': 1
}

# Ваші ідеальні параметри Optuna
cb_params = {
    'iterations': 1787, 'learning_rate': 0.06874094281549652, 'depth': 3,
    'l2_leaf_reg': 5.290355672999665, 'random_strength': 11.502343031921322,
    'bagging_temperature': 0.4122157832620809, 'border_count': 154,
    'eval_metric': 'AUC', 'verbose': False
}

xgb_params = {
    'n_estimators': 2372, 'learning_rate': 0.010081751720435782, 'max_depth': 8,
    'min_child_weight': 5, 'subsample': 0.8434336534358337, 'colsample_bytree': 0.6396772098220731,
    'gamma': 6.228982752620394, 'reg_alpha': 0.0011212515235573147, 'reg_lambda': 0.0008962757039321162,
    'tree_method': 'hist', 'enable_categorical': True,
    'monotone_constraints': mono_constraints, 'random_state': RANDOM_STATE, 'n_jobs': -1
}

# Повертаємо класичні стабільні ліси!
rf_params = {'n_estimators': 800, 'max_depth': 7, 'min_samples_leaf': 4, 'random_state': RANDOM_STATE, 'n_jobs': -1}
et_params = {'n_estimators': 800, 'max_depth': 7, 'min_samples_leaf': 4, 'random_state': RANDOM_STATE, 'n_jobs': -1}

Smoothed TE, Seed Blending та Навчання

In [40]:
N_SPLITS = 5
N_REPEATS = 3
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

oof_cb, test_cb = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))
oof_xgb, test_xgb = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))
oof_rf, test_rf = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))
oof_et, test_et = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))

cb_seeds = [42, 2026, 777]

print("🚀 Починаємо масивне навчання (CB, XGB, RF, ET)...")

for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_train_clean, y)):
    X_tr, y_tr = X_train_clean.iloc[train_idx].copy(), y.iloc[train_idx]
    X_va, y_va = X_train_clean.iloc[valid_idx].copy(), y.iloc[valid_idx]
    X_te = X_test_clean.copy()

    # --- Smoothed Target Encoding для School ---
    weight = 15
    global_mean = y_tr.mean()
    agg = y_tr.groupby(X_tr['School']).agg(['count', 'mean'])
    smoothed_map = (agg['count'] * agg['mean'] + weight * global_mean) / (agg['count'] + weight)

    for df_temp in [X_tr, X_va, X_te]:
        df_temp['School_TE'] = df_temp['School'].map(smoothed_map).fillna(global_mean)
        df_temp.drop(['School'], axis=1, inplace=True)

    # --- 1. CatBoost (Seed Blending) ---
    cat_cols_cb = ['Position_Type', 'Position', 'School_Raw']
    X_tr_cb, X_va_cb, X_te_cb = X_tr.drop(['School_TE'], axis=1), X_va.drop(['School_TE'], axis=1), X_te.drop(['School_TE'], axis=1)

    cb_fold_oof, cb_fold_test = np.zeros(len(valid_idx)), np.zeros(len(X_te))
    for seed in cb_seeds:
        cb_params['random_seed'] = seed
        model_cb = CatBoostClassifier(**cb_params)
        model_cb.fit(Pool(X_tr_cb, y_tr, cat_features=cat_cols_cb),
                     eval_set=Pool(X_va_cb, y_va, cat_features=cat_cols_cb),
                     early_stopping_rounds=100)
        cb_fold_oof += model_cb.predict_proba(X_va_cb)[:, 1] / len(cb_seeds)
        cb_fold_test += model_cb.predict_proba(X_te_cb)[:, 1] / len(cb_seeds)

    oof_cb[valid_idx] += cb_fold_oof / N_REPEATS
    test_cb += cb_fold_test / (N_SPLITS * N_REPEATS)

    # --- 2. XGBoost ---
    X_tr_trees = X_tr.drop(['School_Raw'], axis=1)
    X_va_trees = X_va.drop(['School_Raw'], axis=1)
    X_te_trees = X_te.drop(['School_Raw'], axis=1)

    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr_trees, y_tr, eval_set=[(X_va_trees, y_va)], verbose=False)
    oof_xgb[valid_idx] += model_xgb.predict_proba(X_va_trees)[:, 1] / N_REPEATS
    test_xgb += model_xgb.predict_proba(X_te_trees)[:, 1] / (N_SPLITS * N_REPEATS)

    # --- 3. Random Forest та Extra Trees ---
    X_tr_sk = X_tr_trees.drop(['Position_Type', 'Position'], axis=1).fillna(-999)
    X_va_sk = X_va_trees.drop(['Position_Type', 'Position'], axis=1).fillna(-999)
    X_te_sk = X_te_trees.drop(['Position_Type', 'Position'], axis=1).fillna(-999)

    model_rf = RandomForestClassifier(**rf_params).fit(X_tr_sk, y_tr)
    oof_rf[valid_idx] += model_rf.predict_proba(X_va_sk)[:, 1] / N_REPEATS
    test_rf += model_rf.predict_proba(X_te_sk)[:, 1] / (N_SPLITS * N_REPEATS)

    model_et = ExtraTreesClassifier(**et_params).fit(X_tr_sk, y_tr)
    oof_et[valid_idx] += model_et.predict_proba(X_va_sk)[:, 1] / N_REPEATS
    test_et += model_et.predict_proba(X_te_sk)[:, 1] / (N_SPLITS * N_REPEATS)

🚀 Починаємо масивне навчання (CB, XGB, RF, ET)...


4-D Grid Search Оптимізатор

In [41]:
print("🔍 Запускаємо 4-D Grid Search для пошуку абсолютно ідеальних ваг...")

best_auc = 0
best_weights = [1.0, 0.0, 0.0, 0.0]

for w_cb in np.arange(0, 1.01, 0.01):
    for w_xgb in np.arange(0, 1.01 - w_cb, 0.01):
        for w_rf in np.arange(0, 1.01 - w_cb - w_xgb, 0.01):
            w_et = 1.0 - w_cb - w_xgb - w_rf

            if w_et < -1e-5:
                continue

            blend_oof = (w_cb * oof_cb) + (w_xgb * oof_xgb) + (w_rf * oof_rf) + (w_et * oof_et)
            current_auc = roc_auc_score(y, blend_oof)

            if current_auc > best_auc:
                best_auc = current_auc
                best_weights = [w_cb, w_xgb, w_rf, w_et]

bw = best_weights

print("\n" + "="*40)
print(f"🌲 CatBoost (3 Seeds) OOF AUC: {roc_auc_score(y, oof_cb):.5f}")
print(f"⚡ XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}")
print(f"🌳 Random Forest OOF AUC: {roc_auc_score(y, oof_rf):.5f}")
print(f"🌿 Extra Trees OOF AUC: {roc_auc_score(y, oof_et):.5f}")
print("="*40)
print(f"💎 Знайдені Grid Search ваги: CB={bw[0]:.0%}, XGB={bw[1]:.0%}, RF={bw[2]:.0%}, ET={bw[3]:.0%}")
print(f"🏆 ФІНАЛЬНИЙ ENSEMBLE OOF AUC: {best_auc:.5f}")

🔍 Запускаємо 4-D Grid Search для пошуку абсолютно ідеальних ваг...

🌲 CatBoost (3 Seeds) OOF AUC: 0.84163
⚡ XGBoost OOF AUC: 0.83607
🌳 Random Forest OOF AUC: 0.82483
🌿 Extra Trees OOF AUC: 0.80485
💎 Знайдені Grid Search ваги: CB=73%, XGB=27%, RF=0%, ET=0%
🏆 ФІНАЛЬНИЙ ENSEMBLE OOF AUC: 0.84283


Фінальний Сабміт

In [42]:
# Чітко 4 елементи в масиві bw (CatBoost, XGBoost, RF, ET)
final_test_preds = (bw[0] * test_cb) + (bw[1] * test_xgb) + \
                   (bw[2] * test_rf) + (bw[3] * test_et)

submission = sample_sub.copy()
submission['Drafted'] = final_test_preds
OUTPUT_PATH = DATA_DIR / 'ultimate_revert_to_gold_submission.csv'
submission.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Збережено ідеальний сабміт: {OUTPUT_PATH.name}")

✅ Збережено ідеальний сабміт: ultimate_revert_to_gold_submission.csv
